# Preprocessing of 2023 SKT Living Population Data by Age and Gender

This code aggregates and preprocesses restricted-access SKT living population data into a usable format for socioeconomic vulnerability analysis.

To ensure consistency with the socioeconomic vulnerability data, only records from 2023 were selected. The data were then aggregated by administrative dong, age group, and gender to prepare population indicators for flood vulnerability assessment.

The preprocessing was conducted within the Busan Big Data Center's secure environment.

In [ ]:
import pandas as pd
import os

# Setting
pop008_folder = "D:/20260914_권나영/POP008"
start_period = 202301
end_period = 202312

# Read the csv file
df = pd.read_csv(os.path.join(pop0008_folder, "POP008_202301.csv"), sep="|")
rows, columns = df.shape
print(f"데이터프레임의 행 수: {rows}") # Number of rows in the DataFrame
print(f"데이터프레임의 열 수: {columns}") # Number of columnss in the DataFrame

columns_list = df.columns.tolist()
print("데이터프레임의 컬럼 이름 리스트:", columns_list) # List of column names in the DataFrame


데이터프레임의 행 수: 9446100
데이터프레임의 열 수: 7
데이터프레임의 컬럼 이름 리스트: ['기준일자', '행정동코드', '성연령', '유입지코드', '주거인구', '직장인구', '방문인구']


### Check the original data

In [17]:
df.head()

,기준일자,행정동코드,성연령,유입지코드,주거인구,직장인구,방문인구
0,20230101,26110510,m_0009,11,0,0,13
1,20230101,26110510,m_0009,2611053000,0,0,1
2,20230101,26110510,m_0009,2611054500,0,0,1
3,20230101,26110510,m_0009,2611060000,0,0,1
4,20230101,26110510,m_0009,2614053000,0,0,1


In [20]:
unique_gender_age = df["성연령"].unique()
print("unique_gender_age:\n", unique_gender_age)

unique_gender_age:
 ['m_0009' 'm_1014' 'm_1519' 'm_2024' 'm_5559' 'm_2529' 'w_3539' 'm_6064'
 'w_4044' 'm_3034' 'm_6569' 'w_4549' 'm_7000' 'm_3539' 'w_0009' 'w_1014'
 'w_5054' 'm_4044' 'w_1519' 'm_5054' 'w_2024' 'm_4549' 'w_5559' 'w_2529'
 'w_3034' 'w_6064' 'w_6569' 'w_7000']


In [ ]:
# Example: Filter records by date, administrative dong, and demographic group
sample = df[
    (df["기준일자"] == 20230101) & # Date: January 1, 2023
    (df["행정동코드"] == 26110510) & # Administrative dong code: 26110510
    (df["성연령"] == "m_3034") # Demographic group: Males aged 30–34
]

# Display the first 50 rows of origin codes and population counts
# (residential, working, and visiting populations)
sample[["유입지코드", "주거인구", "직장인구", "방문인구"]].head(50)

### Aggregation of 2023 Living Population Data by Administrative Dong

In [ ]:
import pandas as pd
import os
import gc

# 1. Set the input data directory
folder = r"D:/20260914_권나영/POP008"

# Define demographic groups based on age and gender codes
child = ["m_0009", "w_0009"] # Children: Males and females aged 0–9
old = ["m_6569", "w_6569", "m_7000", "w_7000"] # Elderly: Males and females aged 65 and above

# Initialize a list to store monthly processing results
monthly = []



# 2. Process monthly population data for 2023
for m in range(1, 13):
    ym = f"2023{m:02d}"
    path = os.path.join(folder, f"POP008_{ym}.csv")
    print(ym, "Reading")

    # Read the monthly pipe-delimited CSV file
    df = pd.read_csv(path, sep="|")

    # Calculate total living population
    # as the sum of residential, working, and visiting populations
    df["총생활인구"] = (
        df["주거인구"] + df["직장인구"] + df["방문인구"]
    )

    # Extract population counts for children under 10 and elderly aged 65+
    # Assign zero to records outside each demographic group
    df["10세미만"] = df["총생활인구"].where(
        df["성연령"].isin(child), 0
    )
    df["65세이상"] = df["총생활인구"].where(
        df["성연령"].isin(old), 0
    )

    # Aggregate population counts by date and administrative dong
    # Sum across demographic groups and origin areas
    daily = (
        df.groupby(["기준일자", "행정동코드"])[
            ["주거인구", "직장인구", "방문인구",
             "총생활인구", "10세미만", "65세이상"]
        ]
        .sum()
        .reset_index()
    )

    # Store the daily aggregated data for each month
    monthly.append(daily)

    print(ym, "Completed")



# 3. Calculate annual average daily population
# Combine daily population records from all 12 months
daily_2023 = pd.concat(monthly, ignore_index=True)

# Calculate the mean daily population for each administrative dong and count the number of unique observation dates
result = (
    daily_2023.groupby("행정동코드")
    .agg(
        일평균_주거인구=("주거인구", "mean"),
        일평균_직장인구=("직장인구", "mean"),
        일평균_방문인구=("방문인구", "mean"),
        일평균_총생활인구=("총생활인구", "mean"),
        일평균_10세미만=("10세미만", "mean"),
        일평균_65세이상=("65세이상", "mean"),
        관측일수=("기준일자", "nunique")
    )
    .reset_index()
)



# 4. Calculate demographic population ratios
result["10세미만비율"] = (
    result["일평균_10세미만"] / result["일평균_총생활인구"] # Calculate the proportion of children under 10
)

result["65세이상비율"] = (
    result["일평균_65세이상"] / result["일평균_총생활인구"] # and elderly aged 65+ relative to the total living population
)



# 5. Inspect the final aggregated dataset
# Display the first five rows
print(result.head())

# Check the distribution of observation days across administrative dongs
print(result["관측일수"].value_counts())

202301 Reading
202301 완료
202302 Reading
202302 완료
202303 Reading
202303 완료
202304 Reading
202304 완료
202305 Reading
202305 완료
202306 Reading
202306 완료
202307 Reading
202307 완료
202308 Reading
202308 완료
202309 Reading
202309 완료
202310 Reading
202310 완료
202311 Reading
202311 완료
202312 Reading
202312 완료
      행정동코드     일평균_주거인구     일평균_직장인구     일평균_방문인구     일평균_총생활인구   일평균_10세미만  \
0  26110510  2456.260274  7336.635616  4253.972603  14046.868493  142.079452   
1  26110520  2059.542466   597.819178   746.646575   3404.008219   37.430137   
2  26110530  5009.361644   975.301370  1601.178082   7585.841096  302.564384   
3  26110545  7698.290411   682.520548  1078.326027   9459.136986  416.260274   
4  26110560  3809.353425  1350.287671  4100.000000   9259.641096  221.126027   

     일평균_65세이상  관측일수   10세미만비율   65세이상비율  
0  1622.408219   365  0.010115  0.115500  
1   771.345205   365  0.010996  0.226599  
2  1714.616438   365  0.039885  0.226029  
3  2324.945205   365  0.044006  0.245788  
4  1

### Save

In [25]:
result.to_csv(r"D:/20260914_권나영/POP008_2023_행정동_연간일평균.csv", index=False, encoding="utf-8-sig")